# 🤖 Social Media Post AI Agent
**Workflow:** User request → Company/Product lookup → Reports → Style Guide → Post Generation → Scheduling → Logging

**Setup:** Fill in your API keys in `config.py` before running.

## 📦 Step 0: Imports & Config

In [1]:
import os, json, re, datetime, time
from pathlib import Path

# ── Local modules (same folder as this notebook) ──────────────────────────────
from agent_storage   import Storage
from agent_research  import ResearchAgent
from agent_posts     import PLATFORM_AGENTS, build_context, parse_platforms
from agent_parallel  import ParallelWorkflow, run_all_posts
from agent_schedule  import ScheduleAgent
from agent_logger    import Logger
from agent_utils     import confirm, multiline_input, print_receipt

print("✅ Imports OK")

✅ Imports OK


## 🔑 Step 1: API Keys

In [ ]:
# ── Paste your keys here (or load from .env) ───────────────────────────────────
OPENAI_API_KEY        = os.getenv("OPENAI_API_KEY",  "sk-...")   # GPT-4o for main agents
OPENAI_REVIEW_KEY     = os.getenv("OPENAI_API_KEY",  "sk-...")   # GPT-3.5 for A/B reviewer
SERPER_API_KEY        = os.getenv("SERPER_API_KEY",  "...")       # serper.dev – web search
GOOGLE_CALENDAR_ID    = os.getenv("GCAL_ID",         "primary")   # your calendar id
# Google Calendar credentials JSON path (from Google Cloud Console → Service Account)
GCAL_CREDENTIALS_JSON = os.getenv("GCAL_CREDENTIALS", "credentials.json")

print("✅ Keys loaded")

## 📁 Step 2: Storage Paths

In [ ]:
# Root of the project — change if your folder is elsewhere
PROJECT_ROOT = Path(".")   # same folder as this notebook
storage = Storage(PROJECT_ROOT / "AI Storage")
print(f"📂 AI Storage: {storage.root.resolve()}")

## 🗣️ Step 3: Parse User Request

In [ ]:
# ── EXAMPLE: change this string or make it an input() ─────────────────────────
USER_REQUEST = "Create 3 Instagram posts for Rita's Kiwi Melon in June + Schedule. Make it interactive please."

from agent_utils import parse_request
receipt = parse_request(USER_REQUEST)
print_receipt(receipt)

## 🔍 Step 4: Company & Product Lookup

In [ ]:
researcher = ResearchAgent(
    storage=storage,
    openai_key=OPENAI_API_KEY,
    serper_key=SERPER_API_KEY,
)

company_report, product_report = await researcher.resolve(
    company_name=receipt["company"],
    product_name=receipt["product"],
)

## 📖 Step 5: Review Reports (Optional)

In [ ]:
want_review = confirm("Would you like to review the Company or Product report before continuing? (y/N)")
if want_review:
    choice = input("Type 'company', 'product', or 'both': ").strip().lower()
    if choice in ("company", "both"):
        print(json.dumps(company_report, indent=2))
    if choice in ("product", "both"):
        print(json.dumps(product_report, indent=2))
    input("\nPress Enter when done reviewing (or edit the JSON files directly, then press Enter)…")

## 🎨 Step 6: Style Guide

In [ ]:
style_guide = await researcher.resolve_style_guide(
    company_report=company_report,
    product_report=product_report,
)

## ✅ Step 7: Confirm Receipt

In [ ]:
from agent_utils import interactive_receipt_editor
receipt = interactive_receipt_editor(receipt)
print_receipt(receipt)

## ✍️ Step 8: Generate Posts (Parallel Workflow)

In [ ]:
# ============================================================================
# STEP 8: PARALLEL POST GENERATION
# ============================================================================
# Mirrors the class notebook (07_workflow_multitasking.ipynb) pattern:
#   1. Build a shared context (replaces product_brief from class)
#   2. Instantiate one specialized agent per platform (Instagram/Twitter/Blog)
#   3. Run all agents in parallel using ParallelWorkflow + ThreadPoolExecutor
#   4. Display results in the same bordered format as class output
# ============================================================================

# ── 1. Build shared context from reports (replaces product_brief) ─────────────
context = build_context(
    company=company_report,
    product=product_report,
    style=style_guide,
    extra=receipt.get('additional_info', ''),
    month=receipt.get('when', ''),
)

# ── 2. Build agent list — one per platform, matching class pattern ────────────
platforms  = parse_platforms(receipt.get('platforms', 'instagram'))
n_posts    = int(receipt.get('num_posts', 1))

# For each post number, create one agent per requested platform.
# e.g. 2 posts × [Instagram, Twitter] = 4 agents total across 2 parallel batches
agents_per_post = [
    [
        PLATFORM_AGENTS[p](openai_key=OPENAI_API_KEY, review_key=OPENAI_REVIEW_KEY)
        for p in platforms
        if p in PLATFORM_AGENTS
    ]
    for _ in range(n_posts)
]

print(f'🚀 Starting post generation...\n' + '=' * 70)
print(f'📋 Platforms: {', '.join(p.capitalize() for p in platforms)}')
print(f'   Total posts per platform: {n_posts}')
print(f'   Total agent runs: {n_posts * len(platforms)}')
print('=' * 70 + '\n')

print(f'✅ Initialized agents:')
for agent in agents_per_post[0]:
    print(f'   • {agent.name}')
print()

# ── 3. Run parallel workflow — same as class notebook ─────────────────────────
base_brief = {'context': context, 'total_posts': n_posts}

start_time = time.time()
posts = run_all_posts(agents_per_post, base_brief)
elapsed = time.time() - start_time

print(f'\n✨ All agents finished in {elapsed:.2f} seconds')
print('=' * 70)

# ── 4. Display results — same bordered format as class notebook ───────────────
print('\n📊 GENERATED SOCIAL MEDIA CONTENT')
print('=' * 70 + '\n')

for i, post in enumerate(posts, 1):
    label = f'Post {i} [{post.get("platform","?").upper()}] — Score: {post.get("score", "N/A")}'
    print(f'┌─ {label} ' + '─' * max(0, 68 - len(label)))
    print('│')
    for line in post.get('caption', '').split('\n'):
        print(f'│ {line}')
    print('└' + '─' * 69)
    print()

print('=' * 70)
print(f'✅ Generation complete! Total time: {elapsed:.2f}s')
print('=' * 70)

## 💾 Step 9: Save Posts

In [ ]:
output_dir = storage.save_posts(
    company=receipt["company"],
    product=receipt["product"],
    posts=posts,
    receipt=receipt,
)
print(f"📁 Posts saved to: {output_dir}")

## 📅 Step 10: Schedule Posts (Optional)

In [ ]:
if receipt.get("schedule"):
    scheduler = ScheduleAgent(
        openai_key=OPENAI_API_KEY,
        credentials_json=GCAL_CREDENTIALS_JSON,
        calendar_id=GOOGLE_CALENDAR_ID,
    )
    scheduler.run(
        receipt=receipt,
        posts=posts,
        output_dir=output_dir,
    )
else:
    print("⏭️  Scheduling skipped.")

## 📝 Step 11: Write Log

In [ ]:
logger = Logger(storage)
log_path = logger.write(
    receipt=receipt,
    company_report=company_report,
    product_report=product_report,
    style_guide=style_guide,
    posts=posts,
    output_dir=output_dir,
)
print(f"\n🎉 All done! Log: {log_path}")